pip install rocketpy

In feed system, if we have designed it right behaviour of nitrous oxide should be either a non equilibrium liquid or a low quality saturated liquid vapor. In feed system we want to minimize line pressure losses. If we see high line pressure losses, then we have high cavitation upstream which we want to avoid.

At end we check if our pressure is above P_sat to ensure no cavitation

In [ ]:
import numpy as np
import CoolProp.CoolProp as CP
from rocketprops.rocket_prop import get_prop
from rocketprops.line_supt import calc_line_vel_dp

# --- Unit conversions ---
PA_TO_PSI = 1 / 6894.75729
M_TO_IN = 39.3701
KG_S_TO_LBM_S = 2.20462
K_TO_R = 1.8

# --- Flow param from model ---
T_liquid = 263.15       # liquid temperature from tank [K]
P_upstream = 3e6        # [Pa]
P_downstream = 2.5e6    # [Pa]
m_dot = 1               # [kg/s]

# --- Poorly understood params ---
pipe_roughness = 1e-6   # pipe roughness [m] (guess)
pipe_K_total = 2.3      # estimating with diaphragm valve K constant

# --- Pipe and fluid setup ---
pipe_ID = (3/4)       # pipe inner diameter [in] 
print(f"Pipe Diam: {pipe_ID} (in) ")

pipe_L = 24            # pipe length [in]
print(f"Pipe Length: {pipe_L} (in),  = {0.0254*pipe_L} (m)")

rho_liq = CP.PropsSI('D', 'T', T_liquid, 'P', P_upstream, 'N2O')

# --- rocketprops for darcy weishbach ---
pObj = get_prop('N2O')
vel, deltaP_line = calc_line_vel_dp(
    pObj, 
    T_liquid * K_TO_R, 
    P_upstream * PA_TO_PSI,
    m_dot * KG_S_TO_LBM_S,
    pipe_ID, 
    pipe_roughness* M_TO_IN, 
    pipe_K_total, 
    pipe_L* M_TO_IN
    )

deltaP_line/=PA_TO_PSI

print(f"Upstream pressure: {P_upstream:.2f} Pa, Liquid Density: {rho_liq:.2f}")
print(f"SPI mass flow: {m_dot:.4f} kg/s")
print(f"Pipe pressure drop: {deltaP_line:.2f} Pa (does not include inj pressure drop)")

pcnt_dp = deltaP_line/(P_upstream - P_downstream)*100
print(f"Percent pressure drop (line/(up-down)): {pcnt_dp:.2f} %")


# Determine the phase of nitrous oxide at the downstream conditions
phase = CP.PhaseSI("P", P_downstream, "T", T_liquid, "N2O")

# Check if it’s liquid or not
if phase in ["liquid", "compressed_liquid"]:
    print("P_downstream > P_sat. Liquid at exit, calc valid")
else:
    print(f"INVALID CALC! {phase.upper()} at exit, likely cavitation")

Pipe Diam: 0.5 (in) 
Pipe Length: 24 (in),  = 0.6095999999999999 (m)
Upstream pressure: 3000000.00 Pa, Liquid Density: 958.42
SPI mass flow: 1.0000 kg/s
Pipe pressure drop: 880234.70 Pa (does not include inj pressure drop)
Percent pressure drop (line/(up-down)): 176.05 %
P_downstream > P_sat. Liquid at exit, calc valid
